Wifi data processing
For given wifi data ('updated_densitymap_hourly_cleaned.csv'), process and make some filter actions.

3.1  Exclude building type include 'Residential', 'Parking', 'Housing' 'Commercial'.

3.2  For data recorded in same building, same floor, get the average and max density cnt( density cnt refers to the number of people in the space at a given point in time), and get average and max density ratio.


In [16]:
import pandas as pd
import numpy as np

# READ CSV FILE
df = pd.read_csv('updated_densitymap_hourly_cleaned.csv')
df['building_type'] = df['building_type'].str.strip()

# exclude 'building_type' 列中为  'Parking', 'Housing' 'Commercial' 
filtered_df = df[~df['building_type'].isin([ 'Parking', 'Housing', 'Commercial'])]


# calculate max and mean density cnt based on same building floor and building desc
# cnt refers to the number of people in the space at a given point in time
grouped_values = filtered_df.groupby(['building_floor', 'building_desc'], as_index=False).agg(
    mean_density_cnt=('density_cnt', 'mean'),
    max_density_cnt=('density_cnt', 'max')
)

#only record first column for each same building floor + building description
first_row_flags = filtered_df.groupby(['building_floor', 'building_desc']).cumcount() == 0
representative_rows = filtered_df[first_row_flags]

# add mean and max density into file
final_representative_rows = pd.merge(representative_rows, grouped_values, on=['building_floor', 'building_desc'], how='left')

# delete 'date'  'hour' 'density_cnt''dc_ratio'
final_representative_rows_cleaned = final_representative_rows.drop(columns=['date', 'hour','density_cnt','dc_ratio'], errors='ignore')

# add  'average_dc_ratio' and max dc ratio
final_representative_rows_cleaned['average_dc_ratio'] = final_representative_rows_cleaned['mean_density_cnt'] / final_representative_rows_cleaned['capacity']
 
final_representative_rows_cleaned['max_dc_ratio'] = final_representative_rows_cleaned['max_density_cnt'] / final_representative_rows_cleaned['capacity']

# create processed file
final_representative_rows_cleaned.to_csv('pre_wifi_update.csv', index=False)

final_representative_rows_cleaned.head()

,building_floor,building_desc,latitude,longitude,building_type,capacity,mean_density_cnt,max_density_cnt,average_dc_ratio,max_dc_ratio
0,4,209 BAY STATE ROAD,42.3507,-71.1028,Residential,8,2.284371,9,0.285546,1.125000
1,2,200-202 BAY STATE ROAD,42.3503,-71.1026,Residential,12,4.839326,22,0.403277,1.833333
2,2,184 BAY STATE ROAD,42.3503,-71.1019,Residential,6,2.933566,11,0.488928,1.833333
3,3,190 BAY STATE ROAD,42.3503,-71.1021,Residential,9,2.137515,10,0.237502,1.111111
4,3,161 BAY STATE ROAD,42.3506,-71.1009,Residential,5,2.322030,15,0.464406,3.000000


now continue change the pre_wifi_update.csv. for residential type, only remain Large Dormitory-Style type.

this can be done by using Building and Kitchen List.xlsx.

In [17]:
wifi_df = pd.read_csv('pre_wifi_update.csv')

# read excel
building_list_df = pd.read_excel('Building and Kitchen List.xlsx')

# Identify all housing types labeled "Large Dormitory Style" from the Excel file
large_dormitory_buildings = building_list_df[building_list_df['Residential Building Type'] == 'Large Dormitory-Style']['Building Description'].unique()
# Add the "Residential Building Type" column and set it to an empty string by default
wifi_df['Residential Building Type'] = ''

# Set "Residential Building Type" for rows that meet the criteria
wifi_df.loc[wifi_df['building_type'].str.contains('Residential') & wifi_df['building_desc'].isin(large_dormitory_buildings), 'Residential Building Type'] = 'Large Dormitory-Style'

# Filter data in wifi_update.csv
# Keep all rows for non-residential types, for residential types, only those that belong to the "Large Dormitory Style"
filtered_wifi_df = wifi_df[
    (~wifi_df['building_type'].str.contains('Residential')) |
    (wifi_df['building_type'].str.contains('Residential') & wifi_df['building_desc'].isin(large_dormitory_buildings))
]
output_file_path = 'wifi_update.csv'

filtered_wifi_df.to_csv(output_file_path, index=False)
filtered_wifi_df.head

<bound method NDFrame.head of      building_floor                  building_desc  latitude  longitude  \
8                 1        771 COMMONWEALTH AVENUE   42.3510   -71.1080   
9                 2        771 COMMONWEALTH AVENUE   42.3510   -71.1080   
10                1        775 COMMONWEALTH AVENUE   42.3509   -71.1090   
11                1              1 UNIVERSITY ROAD   42.3512   -71.1100   
12                1  745 - 755 COMMONWEALTH AVENUE   42.3505   -71.1073   
...             ...                            ...       ...        ...   
1134              3    617-621 COMMONWEALTH AVENUE   42.3497   -71.1013   
1135              p             100 BAY STATE ROAD   42.3498   -71.0979   
1136             81              91 BAY STATE ROAD   42.3503   -71.0970   
1138              A        700 COMMONWEALTH AVENUE   42.3493   -71.1040   
1139              W             140 BAY STATE ROAD   42.3500   -71.0998   

        building_type  capacity  mean_density_cnt  max_density_cnt  \

In [18]:
#same as what houjie does in teamwork.ipynb
# read csv file
df = pd.read_csv('wifi_update.csv')

# delete p in building floor becuase we do not consider it
df = df[~df['building_floor'].str.contains('p', case=False)]

#add 'Maintained by BU FMO'from Building and Kitchen List.xlsx做
# read excel
building_list_df = pd.read_excel('Building and Kitchen List.xlsx')
# Merge two dataframes based on 'Building Description', adding 'Maintained by BU FMO' information to 'wifi_update' DataFrame
df_merged = pd.merge(df, building_list_df[['Building Description', 'Maintained by BU FMO']], 
                     left_on='building_desc', right_on='Building Description', 
                     how='left')

# Check the number of empty values in the 'Maintained by BU FMO' column in the merged result due to possible mismatch
null_counts = df_merged['Maintained by BU FMO'].isnull().sum()


#now saved as wifi_teamworkupdate.csv
df_merged.to_csv('wifi_teamworkupdate.csv', index=False)
df_merged.head()

,building_floor,building_desc,latitude,longitude,building_type,capacity,mean_density_cnt,max_density_cnt,average_dc_ratio,max_dc_ratio,Residential Building Type,Building Description,Maintained by BU FMO
0,1,771 COMMONWEALTH AVENUE,42.3510,-71.1080,Student Support,73,25.448998,309,0.348616,4.232877,NaN,771 COMMONWEALTH AVENUE,YES
1,2,771 COMMONWEALTH AVENUE,42.3510,-71.1080,Student Support,92,30.477226,283,0.331274,3.076087,NaN,771 COMMONWEALTH AVENUE,YES
2,1,775 COMMONWEALTH AVENUE,42.3509,-71.1090,Student Support,534,78.563533,534,0.147123,1.000000,NaN,775 COMMONWEALTH AVENUE,YES
3,1,1 UNIVERSITY ROAD,42.3512,-71.1100,Academic,307,51.585595,351,0.168031,1.143322,NaN,1 UNIVERSITY ROAD,YES
4,1,745 - 755 COMMONWEALTH AVENUE,42.3505,-71.1073,Academic,129,40.219413,319,0.311778,2.472868,NaN,745 - 755 COMMONWEALTH AVENUE,YES
